In [4]:
# 라이브러리 설치
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.3 MB/s eta 0:00:00


In [2]:

import torch, gc
import os, re, json, glob, csv, random, glob
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
from g2pk import G2p
from faster_whisper import WhisperModel
from rapidfuzz.distance import Levenshtein
from rapidfuzz import process, fuzz
from mecab import MeCab


In [ ]:
#그래픽카드 메모리 남용을 막기 위한 캐시 초기화

gc.collect()
torch.cuda.empty_cache()

In [3]:


#Googledrive 마운트
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [17]:
# ==============================================================================
# 1. 환경 설정 및 하이퍼파라미터
# ==============================================================================

# 1-1. 경로 설정
BASE_PROJECT_PATH = "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results"
if not os.path.exists(BASE_PROJECT_PATH): #경로 존재 확인
    os.makedirs(BASE_PROJECT_PATH, exist_ok=True)

#1-1. 기본 파일 경로 변수
AUDIO_FOLDER      = os.environ.get("AUDIO_FOLDER", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/voice_record")
TRANSCRIPTS_PATH  = os.environ.get("TRANSCRIPTS_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/transcripts.json")
BIAS_PATH         = os.environ.get("BIASING_LIST_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/biasing_list.json")
EPG_PATH          = os.environ.get("EPG_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/epg.json")
CATALOG_PATH      = os.environ.get("CATALOG_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/catalog.json")

# 1-2. 실험 변수 (Hyper-parameters)
HOTWORD_TOPK_SWEEP = [20, 30, 50]      # Hotwords 개수 SWEEP
BIAS_ITERATION_SWEEP = [5,10]       # 가중치 업데이트 주기 SWEEP
POSTPROCESS_SWEEP = [1]        # 후처리 적용 여부 (0: OFF, 1 : ON)

# ==============================================================================
# 1-3. 후처리 및 임계치 설정 (Post-processing & Thresholds)
# 설명: ASR 인식 결과를 DB 내 고유명사로 교정하기 위한 로직의 판단 기준입니다.
# ==============================================================================

# 화이트리스트 후보군 추출 개수 (Fuzzy Matching을 수행할 상위 N개의 후보)
WL_TOPN = int(os.environ.get("WL_TOPN", "80"))
# 전체 문장 유사도 임계치 (0~100): 이 점수 이상일 때만 교정 로직이 작동 (엄격한 기준: 92)
RULE_WRATIO_TH = int(os.environ.get("RULE_WRATIO_TH", "92"))
# 부분 문자열 오차율 허용치 (Gate): 교체 구간의 CER이 이 수치 이하일 때만 최종 교정 (과교정 방지용)
RULE_GATE = float(os.environ.get("RULE_GATE", "0.34"))
# 탐색 길이 허용 오차: 정답 단어 길이 대비 앞뒤로 탐색할 문자의 여유분
RULE_TOL = int(os.environ.get("RULE_TOL", "2"))


# ==============================================================================
# 1-4. ASR 모델 엔진 설정 (Faster-Whisper Configuration)
# 설명: 음성 인식 모델의 크기, 연산 장치 및 추론 정밀도를 정의합니다.
# ==============================================================================

# 사용할 Whisper 모델 규모 (medium: 한국어 인식 속도와 정확도의 최적 균형점)
ASR_MODEL = os.environ.get("ASR_MODEL", "medium")
# 연산 장치 설정 (cuda: NVIDIA GPU 가속 사용, cpu: CPU 사용)
ASR_DEVICE = os.environ.get("ASR_DEVICE", "cuda")
# 연산 정밀도 (float32: 표준 정밀도, float16: 메모리 절약 및 속도 향상 가능)
ASR_COMPUTE = os.environ.get("ASR_COMPUTE", "float32")
# 인식 대상 언어 코드 (ko: 한국어 전용 모드로 설정하여 인식률 최적화)
ASR_LANG = os.environ.get("ASR_LANG", "ko")
# 빔 서치(Beam Search) 크기: 추론 시 검토할 상위 경로 수 (값이 클수록 정확하나 속도 저하)
ASR_BEAM = int(os.environ.get("ASR_BEAM", "5"))

# 1-5. 결과 저장 경로
OUT_ROWS = "./asr_detail.csv"
OUT_SUM  = "./asr_summary.csv"

/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results


In [20]:
# ==============================================================================
# 2. 텍스트 정규화 및 평가 도구 (Text Normalization & Metrics)
# ==============================================================================

class TextNormalizer:
    def __init__(self):
        self.g2p = G2p()

        #알파벳 -> 한글 변환 맵
        self.char_map = {
            "a": "에이", "b": "비", "c": "씨", "d": "디", "e": "이", "f": "에프", "g": "지",
            "h": "에이치", "i": "아이", "j": "제이", "k": "케이", "l": "엘", "m": "엠",
            "n": "엔", "o": "오", "p": "피", "q": "큐", "r": "알", "s": "에스",
            "t": "티", "u": "유", "v": "브이", "w": "더블유", "x": "엑스", "y": "와이", "z": "제트"
        }
        #한글 숫자 발음 단일화 맵
        self.num_sense_map = {"하나": "일", "한": "일", "둘": "이", "두": "이", "셋": "삼", "세": "삼", "여덟": "팔", "열": "십"}

    #숫자 한글 변환 
    def _num_to_ko(self, num_str: str) -> str:
        try:
            n = int(num_str)
            if n == 0: return "영"
            u, t, h = ["","일","이","삼","사","오","육","칠","팔","구"], ["","십","이십","삼십","사십","오십","육십","칠십","팔십","구십"], ["","백","이백","삼백","사백","오백","육백","칠백","팔백","구백"]
            if n >= 100:
                hv, r = divmod(n, 100); tv, uv = divmod(r, 10)
                return h[hv] + (t[tv] if tv != 1 else "십") + u[uv]
            elif n >= 10:
                tv, uv = divmod(n, 10)
                return (t[tv] if tv != 1 else "십") + u[uv]
            return u[n]
        except: return num_str

    #텍스트 정규화(영어, 숫자 -> 한글 변환 및 띄어쓰기 제거)
    def normalize(self, text: str, remove_space: bool = False) -> str:
        if not text: return ""
        s = str(text).lower().strip()

        #영어, 숫자 -> 한글 변환
        s = re.sub(r'\d+', lambda m: self._num_to_ko(m.group()), s)
        for k, v in self.num_sense_map.items():
            s = re.sub(rf'\b{k}\b', v, s)
        for eng, ko in self.char_map.items():
            s = s.replace(eng, ko)

        #remove_space 값에 따라 띄어쓰기 제거
        if remove_space:
            s = re.sub(r"[^0-9\uac00-\ud7a3]", "", s)
        else:
            s = re.sub(r"[^0-9\uac00-\ud7a3\s]", "", s)
            s = re.sub(r"\s+", " ", s).strip()
        return s

# CER, WER 계산 함수
def calculate_cer(ref: str, hyp: str) -> float:
    r = normalizer.normalize(ref, remove_space=True)
    h = normalizer.normalize(hyp, remove_space=True)
    if not r: return 0.0 if not h else 1.0
    return Levenshtein.distance(r, h) / len(r)

def calculate_wer(ref: str, hyp: str) -> float:
    r_text = normalizer.normalize(ref, remove_space=True)
    h_text = normalizer.normalize(hyp, remove_space=True)
    r_morphs, h_morphs = mecab.morphs(r_text), mecab.morphs(h_text)
    if not r_morphs: return 0.0 if not h_morphs else 1.0
    return Levenshtein.distance(r_morphs, h_morphs) / len(r_morphs)

#인식된 고유명사를 정답 고유명사와 비교해 인식 결과를 교정하는 데 도움을 주는 함수
def best_proper_noun_match(entity: str, hyp: str, RULE_TOL: int = 2) -> Tuple[float, str]:
    e = normalizer.normalize(entity, remove_space=True)
    h = normalizer.normalize(hyp, remove_space=True)
    if not e: return 0.0, ""
    if not h: return 1.0, ""

    L = len(e)
    if len(h) <= L: return Levenshtein.distance(e, h) / L, h

    best_score, best_sub = 1.0, ""
    for wlen in range(max(1, L - RULE_TOL), min(len(h), L + RULE_TOL) + 1):
        for i in range(0, len(h) - wlen + 1):
            sub = h[i:i + wlen]
            score = Levenshtein.distance(e, sub) / L
            if score < best_score:
                best_score, best_sub = score, sub
                if best_score == 0.0: return 0.0, best_sub
    return best_score, best_sub

#인식된 고유명사의 recall, avg_pn_cer, matched_texts(교정된 인식 고유명사) 값 반환
def evaluate_proper_nouns(entities: List[str], hyp: str, threshold: float = 0.2) -> Tuple[float, float, List[str]]:
    if not entities: return 0.0, 0.0, []
    results = [best_proper_noun_match(e, hyp) for e in entities]
    cers = [res[0] for res in results]
    matched_texts = [res[1].replace(" ", "") for res in results]
    recall = sum(1 for c in cers if c <= threshold) / len(cers)
    avg_pn_cer = sum(cers) / len(cers)
    return recall, avg_pn_cer, matched_texts


# ==============================================================================
# 3. 적응형 바이어싱 매니저 (Adaptive Bias Manager)
# ==============================================================================

class BiasManager:
    def __init__(self, db_path: str):
        self.db_path = db_path
        with open(db_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.ref_count = self.data.get("ref_count", 0)
        self.data["ref_count"] += 1
        self.session_hits = {}

    def get_weighted_hotwords(self, top_k: int) -> List[str]:
        if top_k <= 0: return []
        words = list(self.data["global"].keys())
        weights = [float(self.data["global"][w]) + 1.0 for w in words]
        return random.choices(words, weights=weights, k=min(top_k, len(words)))

    def add_hit(self, matched_entities: List[str]):
        for ent in matched_entities:
            if ent in self.data["global"]:
                self.session_hits[ent] = self.session_hits.get(ent, 0) + 1

    def finalize(self):
        # 학습 주기에 도달했을 때만 파일 업데이트
        if self.ref_count > 0 and self.ref_count % BIAS_ITERATION_SWEEP == 0:
            print(f"\n[LEARNING] {BIAS_ITERATION_SWEEP}회 주기 학습 실행 (현재 {self.ref_count}회차). 가중치를 갱신합니다.")
            for word, count in self.session_hits.items():
                self.data["global"][word] += count

            with open(self.db_path, "w", encoding="utf-8") as f:
                json.dump(self.data, f, ensure_ascii=False, indent=2)
            print("[LEARNING] 가중치 데이터베이스 저장 완료.")
        else:
            # 주기가 아닐 때는 참조 횟수만 업데이트
            with open(self.db_path, "w", encoding="utf-8") as f:
                json.dump(self.data, f, ensure_ascii=False, indent=2)
            print(f"\n[SESSION] 참조 횟수 기록 ({self.ref_count}회). 가중치 업데이트 대기 중.")


# ==============================================================================
# 4. 데이터 로드 및 후처리 함수
# ==============================================================================

def load_json(path: str) -> Dict[str, Any]:
    if not path or not os.path.isfile(path): return {}
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def load_transcripts(path: str) -> Dict[str, Dict[str, Any]]:
    data = load_json(path)
    out = {}
    for k, v in (data or {}).items():
        if isinstance(v, str): out[k] = {"text": v, "entities": []}
        elif isinstance(v, dict): out[k] = {"text": (v.get("text") or "").strip(), "entities": v.get("entities", []) or []}
        else: out[k] = {"text": "", "entities": []}
    return out

def dedup_clean(words: List[str]) -> List[str]:
    seen, out = set(), []
    for w in words or []:
        w = (w or "").strip()
        if not w or len(w) < 2: continue
        if w in seen: continue
        seen.add(w)
        out.append(w)
    return out

# def epg_titles(epg: Dict[str, Any]) -> List[str]:
#     titles = []
#     for section in ("now", "today"):
#         for item in epg.get(section, []) or []:
#             if isinstance(item, dict) and item.get("title"): titles.append(item["title"])
#             elif isinstance(item, str): titles.append(item)
#     return dedup_clean(titles)

# def catalog_titles(catalog: Dict[str, Any]) -> List[str]:
#     if not isinstance(catalog, dict) or not catalog: return []
#     if isinstance(catalog.get("titles"), list): return dedup_clean([x for x in catalog["titles"] if isinstance(x, str)])
#     out = []
#     for _, v in catalog.items():
#         if isinstance(v, list): out += [x for x in v if isinstance(x, str)]
#     return dedup_clean(out)

# def make_whitelist(hyp_raw: str, epg: Dict[str, Any], catalog: Dict[str, Any], top_n: int = 80) -> List[str]:
#     pool = dedup_clean(epg_titles(epg) + catalog_titles(catalog))
#     if not pool: return []
#     scored = process.extract(hyp_raw, pool, scorer=fuzz.WRatio, limit=min(top_n, len(pool)))
#     return [t for t, _, _ in scored]

def build_norm_with_map(raw: str) -> Tuple[str, List[int]]:
    if not raw: return "", []
    raw_l = raw.lower()
    norm_chars, idx_map = [], []
    for i, ch in enumerate(raw_l):
        if re.match(r"[0-9a-z\u3131-\u318e\uac00-\ud7a3]", ch):
            norm_chars.append(ch)
            idx_map.append(i)
    return "".join(norm_chars), idx_map

# def hotwords_from_context(bias: Dict[str, Any], epg: Dict[str, Any], catalog: Dict[str, Any], top_k: int) -> List[str]:
#     """
#     [Context Injection] 현재 상황에 맞춰 ASR에게 힌트로 줄 단어장 생성
#     - 전체 리스트에서 중복을 제거한 후, 랜덤하게 top_k개를 선정하여 반환합니다.
#     """
#     if top_k <= 0:
#         return []

#     # 1. 전체 후보 단어 풀 생성 (중복 제거 포함)
#     pool = dedup_clean(
#         epg_titles(epg)
#         + coerce_terms(bias.get("global"))
#         + catalog_titles(catalog)
#     )

#     # 2. 리스트보다 요청한 개수(top_k)가 많을 경우를 대비한 예외 처리
#     actual_k = min(top_k, len(pool))

#     # 3. 랜덤 샘플링 수행 (순서 섞임 효과 포함)
#     return random.sample(pool, actual_k)



def best_substring_span_raw(entity: str, hyp_raw: str, tol: int = 2) -> Tuple[Optional[int], Optional[int], float]:
    # [수정] normalizer.normalize(entity)로 호출해야 함 (기존: normalizer(entity))
    e = normalizer.normalize(entity)
    if not e: return None, None, 1.0
    h_norm, h_map = build_norm_with_map(hyp_raw)
    if not h_norm: return None, None, 1.0

    L = len(e)
    if len(h_norm) <= L:
        span_cer = Levenshtein.distance(e, h_norm) / L
        s_raw = h_map[0] if h_map else 0
        e_raw = (h_map[-1] + 1) if h_map else len(hyp_raw)
        return s_raw, e_raw, span_cer

    best = 1.0
    best_s_norm = 0
    best_e_norm = min(len(h_norm), L)
    for wlen in range(max(1, L - tol), min(len(h_norm), L + tol) + 1):
        for i in range(0, len(h_norm) - wlen + 1):
            sub = h_norm[i:i + wlen]
            span_cer = Levenshtein.distance(e, sub) / L
            if span_cer < best:
                best = span_cer
                best_s_norm = i
                best_e_norm = i + wlen
                if best == 0.0: break
        if best == 0.0: break

    s_raw = h_map[best_s_norm]
    e_raw = h_map[best_e_norm - 1] + 1
    return s_raw, e_raw, best

# def postprocess_rule_whitelist(hyp_raw: str, whitelist: List[str], wratio_th: int = 92, gate: float = 0.34, tol: int = 2) -> Tuple[str, List[Dict[str, Any]]]:
#     if not hyp_raw or not whitelist: return hyp_raw, []
#     target = re.sub(r"(틀어(줘)?|재생(해(줘)?)?|보여(줘)?|켜(줘)?|해(줘)?|바꿔(줘)?|변경(해(줘)?)?|채널|좀|지금|다시|tv|티비|전원)", " ", (hyp_raw or "").lower())
#     target = re.sub(r"\s+", " ", target).strip()

#     best = process.extractOne(target if len(target) >= 2 else hyp_raw, whitelist, scorer=fuzz.WRatio)
#     if not best: return hyp_raw, []
#     chosen, wr_score, _ = best

#     if wr_score < wratio_th:
#         return hyp_raw, [{"type": "skip", "reason": "wratio<th", "wratio": float(wr_score), "chosen": chosen, "target": target}]

#     s, e, span_cer = best_substring_span_raw(chosen, hyp_raw, tol=tol)
#     if s is None or e is None or span_cer > gate:
#         return hyp_raw, [{"type": "skip", "reason": "gate_fail", "wratio": float(wr_score), "chosen": chosen, "span_cer": float(span_cer), "target": target}]

#     surface = hyp_raw[s:e]
#     if surface == chosen:
#         return hyp_raw, [{"type": "noop", "wratio": float(wr_score), "chosen": chosen, "span_cer": float(span_cer), "target": target}]

#     hyp_pp = hyp_raw[:s] + chosen + hyp_raw[e:]
#     replog = [{"type": "replace", "start": int(s), "end": int(e), "from": surface, "to": chosen, "wratio": float(wr_score), "span_cer": float(span_cer), "target": target}]
#     return hyp_pp, replog


# ==============================================================================
# 5. ASR 모델 및 결과 집계
# ==============================================================================

@dataclass
class ASR:
    def __init__(self):
        self.model = WhisperModel(ASR_MODEL, device=ASR_DEVICE, compute_type=ASR_COMPUTE)

    def transcribe(self, path: str, hotwords: Optional[List[str]] = None) -> str:
        korean_only_prompt = "엠비씨 뉴스데스크, 티브이엔 유퀴즈, 넷플릭스 파친코 틀어줘, 볼륨 십으로 올려줘, 삼십 분 뒤에 티비 꺼줘, 채널 이십이 번으로 바꿔줘, 에이, 비, 씨, 디, 하나, 둘, 셋, 삼십 초, 오 분, 세 칸, 일 배속"
        kwargs: Dict[str, Any] = {"language": self.cfg.language, "beam_size": self.cfg.beam_size, "initial_prompt": korean_only_prompt, "vad_filter": True}
        if hotwords and len(hotwords) > 0: kwargs["hotwords"] = ",".join(hotwords)
        segs, _ = self.model.transcribe(path, **kwargs)
        return "".join(s.text for s in segs).strip()

def summarize(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    agg: Dict[Tuple[int, int], Dict[str, Any]] = {}
    for r in rows:
        key = (int(r["top_k"]), int(r["preprocess_on"]))
        a = agg.setdefault(key, {"top_k": key[0], "preprocess_on": key[1], "files_num": 0, "cer_sum": 0.0, "wer_sum": 0.0, "pn_r_sum": 0.0, "pn_c_sum": 0.0, "pn_n": 0})
        a["files_num"] += 1
        a["cer_sum"] += float(r["cer"])
        a["wer_sum"] += float(r["wer"])
        if r.get("pn_recall") is not None:
            a["pn_r_sum"] += float(r["pn_recall"])
            a["pn_c_sum"] += float(r["pn_cer"])
            a["pn_n"] += 1

    out = []
    for k, a in sorted(agg.items()):
        pn_n = a["pn_n"]
        out.append({
            "top_k": a["top_k"], "preprocess_on": a["preprocess_on"], "files_num": a["files_num"],
            "cer_mean": a["cer_sum"] / max(1, a["files_num"]),
            "wer_mean": a["wer_sum"] / max(1, a["files_num"]),
            "pn_recall_mean": (a["pn_r_sum"] / pn_n) if pn_n else None,
            "pn_cer_mean": (a["pn_c_sum"] / pn_n) if pn_n else None
        })
    return out


# ==============================================================================
# 6. 메인 실행 (Main Execution)
# ==============================================================================


# 전역 인스턴스 생성
normalizer = TextNormalizer()
mecab = MeCab()

assert os.path.isdir(AUDIO_FOLDER), f"오디오 폴더 없음: {AUDIO_FOLDER}"
assert os.path.isfile(TRANSCRIPTS_PATH), f"transcripts.json 없음: {TRANSCRIPTS_PATH}"

transcripts = load_transcripts(TRANSCRIPTS_PATH)
bias_mgr = BiasManager(BIAS_PATH)
epg = load_json(EPG_PATH)
catalog = load_json(CATALOG_PATH)
files = glob.glob(os.path.join(AUDIO_FOLDER, "**/*.mp4"), recursive=True)
assert files, f"오디오 없음: {AUDIO_FOLDER}"

cfg = ASRConfig(model_size=ASR_MODEL, device=ASR_DEVICE, compute_type=ASR_COMPUTE, language=ASR_LANG, beam_size=ASR_BEAM)
asr = ASR(cfg)

rows: List[Dict[str, Any]] = []  # [수정] 결과 데이터를 저장할 리스트

for top_k in HOTWORD_TOPK_SWEEP:
    for repeat in range(BIAS_ITERATION_SWEEP):
        current_hotwords = bias_mgr.get_weighted_hotwords(top_k)

        for pp_on in POSTPROCESS_SWEEP:
            print(f"\n[RUN] Top-K: {top_k} | Iteration: {repeat+1}/{BIAS_ITERATION_SWEEP} | PostProcess: {pp_on}")

            for audio_path in files:
                fname = os.path.basename(audio_path)
                meta = transcripts.get(fname, {"text": "", "entities": []})

                # ASR & Post-process
                hyp_raw = asr.transcribe(audio_path, hotwords=current_hotwords)
                wl = make_whitelist(hyp_raw, epg=epg, catalog=catalog, top_n=WL_TOPN)

                if pp_on and wl:
                    hyp_final, replog = postprocess_rule_whitelist(hyp_raw, wl, wratio_th=RULE_WRATIO_TH, gate=RULE_GATE, tol=RULE_TOL)
                else:
                    hyp_final, replog = hyp_raw, []

                # Metrics
                cer = calculate_cer(meta["text"], hyp_final)
                wer = calculate_wer(meta["text"], hyp_final)
                pn_recall, pn_cer, recognized_ents = evaluate_proper_nouns(meta["entities"], hyp_final)

                # Hit Counting & Logging
                bias_mgr.add_hit(recognized_ents)
                pn_recall = pn_recall if pn_recall is not None else 0.0

                print(f"- {os.path.dirname(audio_path).split('/')[-1]}/{fname} | cer={cer:.4f} | wer={wer:.4f} | pn_recall={pn_recall:.4f}")
                # 로그 출력

                print(f"- {os.path.dirname(audio_path).split("/")[-1]}/{fname} | pp_on={pp_on} | cer={cer:.4f} | wer={wer:.4f} | pn_c={pn_cer} | pn_recall={pn_recall:.4f}")

                print(f"ref: {meta["text"]}\nhyp_raw: {normalizer.normalize(hyp_raw)}\nref_pp: {meta["entities"]}\nhyp_pp: {recognized_ents}\n")

                # 결과 행 저장
                # [수정] 결과를 rows 리스트에 저장
                rows.append({
                    "file": f'{os.path.dirname(audio_path).split("/")[-1]}/{fname}',
                    "top_k": top_k, "preprocess_on": pp_on,
                    "hotwords": current_hotwords, "hotwords_n": len(current_hotwords),
                    "cer": cer, "wer": wer, "pn_recall": pn_recall, "pn_cer": pn_cer,
                    "ref": meta["text"], "hyp_raw": normalizer.normalize(hyp_raw), "hyp_pp": normalizer.normalize(hyp_final),
                    "ref_pp": meta["entities"], "recognized_pn": recognized_ents,
                    "wl_size": len(wl), "wl_top5": json.dumps(wl[:5], ensure_ascii=False),
                    "replog": json.dumps(replog, ensure_ascii=False)
                })

# 학습 종료 후 최종 업데이트
bias_mgr.finalize()

# 결과 저장
assert rows, "저장할 결과 데이터가 없습니다."

with open(OUT_ROWS, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

summary = summarize(rows)
with open(OUT_SUM, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
    w.writeheader()
    w.writerows(summary)

print("\n[DONE] 실행 및 저장 완료")
print(f"- 상세 결과: {OUT_ROWS}")
print(f"- 요약 결과: {OUT_SUM}")
# ==============================================================================
# [부록] 기존 코드에서 주석 처리된 NLU 및 확장 로직
# ==============================================================================

# 1. 알파벳 및 도메인 특화 단어 발음 매핑 (영어가 포함된 경우 대비)
# ------------------------------------------------------------------------------
# ALPHABET_TO_KO = {
#      "a": "에이", "b": "비", "c": "씨", "d": "디", "e": "이", "f": "에프", "g": "지",
#      "h": "에이치", "i": "아이", "j": "제이", "k": "케이", "l": "엘", "m": "엠",
#      "n": "엔", "o": "오", "p": "피", "q": "큐", "r": "알", "s": "에스",
#      "t": "티", "u": "유", "v": "브이", "w": "더블유", "x": "엑스", "y": "와이", "z": "제트"
# }

# ENG_WORD_TO_KO = {
#      "netflix": "넷플릭스", "youtube": "유튜브", "disney": "디즈니",
#      "tving": "티빙", "apple": "애플", "plus": "플러스", "tv": "티비"
# }


# 2. NLU 핵심 정의 (의도 및 동의어 사전)
# ------------------------------------------------------------------------------
# INTENTS = {
#      "POWER_OFF": "전원 끄기", "POWER_ON": "전원 켜기",
#      "VOLUME_SET": "볼륨 설정", "VOLUME_UP": "볼륨 올리기", "VOLUME_DOWN": "볼륨 내리기",
#      "CHANNEL_TUNE": "채널 변경", "APP_OPEN": "앱 실행", "CONTENT_PLAY": "콘텐츠 재생",
#      "UNKNOWN": "미분류",
# }

# APP_ALIASES = {
#      "유튜브": ["유튜브", "youtube", "you tube"],
#      "넷플릭스": ["넷플릭스", "netflix"],
#      "디즈니플러스": ["디즈니플러스", "disney plus", "disney+", "디즈니+"],
#      "티빙": ["티빙", "tving", "tv ing", "tv-ing"],
# }

# CHANNEL_ALIASES = {
#      "MBC":  ["엠비씨", "mbc", "엠비시"],
#      "SBS":  ["에스비에스", "sbs"],
#      "JTBC": ["제이티비씨", "jtbc"],
#      "KBS1": ["케이비에스1", "kbs1", "케이비에스 1", "kbs 1"],
#      "KBS2": ["케이비에스2", "kbs2", "케이비에스 2", "kbs 2"],
#      "TVN":  ["티비엔", "tvn", "티브이엔", "tv n"],
#      "YTN":  ["와이티엔", "ytn"],
# }


# 3. NLU 정보 추출 함수 (App, Channel 추출)
# ------------------------------------------------------------------------------
# def channel_terms_for_hotwords() -> List[str]:
#      """Hotwords(힌트)에 채널명들도 추가하여 인식률 향상"""
#      terms: List[str] = []
#      for ch_id, keys in (CHANNEL_ALIASES or {}).items():
#          if ch_id and isinstance(ch_id, str):
#              terms.append(ch_id)
#          for k in keys or []:
#              if isinstance(k, str) and k.strip():
#                  terms.append(k.strip())
#      return dedup_clean(terms)

# def extract_app(text: str) -> Optional[str]:
#      """텍스트에서 앱 이름 추출 (동의어 처리 포함)"""
#      if not text: return None
#      for app, keys in APP_ALIASES.items():
#          if _contains_any(text, keys):
#              return app
#      return None

# def extract_channel(text: str) -> Optional[str]:
#      """텍스트에서 채널명 추출 (정규화 포함)"""
#      if not text: return None
#      tl = (text or "").lower()
#      tl = re.sub(r"\s+", " ", tl).strip()
#      tl_ns = tl.replace(" ", "")
#      for ch_id, keys in CHANNEL_ALIASES.items():
#          for k in keys:
#              k_l = k.lower()
#              if k_l in tl or k_l.replace(" ", "") in tl_ns:
#                  return ch_id
#      return None


# 4. NLU 파서 및 성능 측정 지표
# ------------------------------------------------------------------------------
# def nlu_parse(text: str, catalog_pool: List[str]) -> Dict[str, Any]:
#      """텍스트를 입력받아 Intent(의도)와 Slot(상세정보) 반환"""
#      out = {"intent": "UNKNOWN", "slots": {}}
#      if not text: return out
#      tl = text.lower()
#      # 1) 전원 제어
#      if ("꺼" in tl or "끄" in tl) and ("tv" in tl or "티비" in tl or "전원" in tl):
#          out["intent"] = "POWER_OFF"; return out
#      # 2) 볼륨 제어
#      if "볼륨" in tl or "volume" in tl:
#          n = extract_int_number_0_10(text)
#          if n is not None:
#              out["intent"] = "VOLUME_SET"; out["slots"]["volume"] = int(n); return out
#          if "올" in tl or "키" in tl or "높" in tl:
#              out["intent"] = "VOLUME_UP"; return out
#      # 3) 채널 제어
#      ch = extract_channel(text)
#      if ch and (any(x in tl for x in ["채널", "바꿔", "변경", "틀", "켜"])):
#          out["intent"] = "CHANNEL_TUNE"; out["slots"]["channel_id"] = ch; return out
#      # 4) 앱 실행
#      app = extract_app(text)
#      if app and (any(x in tl for x in ["켜", "열", "실행", "틀"])):
#          out["intent"] = "APP_OPEN"; out["slots"]["app"] = app; return out
#      return out

# def intent_slot_metrics(ref_intent, ref_slots, hyp_intent, hyp_slots):
#      """NLU 결과 채점 (정답과 비교)"""
#      if not ref_intent: return None, None
#      intent_acc = 1 if (ref_intent == hyp_intent) else 0
#      if not ref_slots or not isinstance(ref_slots, dict): return intent_acc, None
#      keys = list(ref_slots.keys())
#      if not keys: return intent_acc, 1.0
#      ok = sum(1 for k in keys if k in hyp_slots and hyp_slots.get(k) == ref_slots.get(k))
#      return intent_acc, ok / len(keys)


# 5. 메인 루프 내 NLU 실행 부분
# ------------------------------------------------------------------------------
# nlu = {"intent": None, "slots": {}}
# intent_acc, slot_acc = None, None
# if ENABLE_NLU:
#      nlu = nlu_parse(hyp_pp, content_pool)
#      intent_acc, slot_acc = intent_slot_metrics(ref_intent, ref_slots, nlu["intent"], nlu.get("slots", {}))


[RUN] Top-K: 20 | Iteration: 1/5 | PostProcess: 0
- lee-eunje/record14.mp4 | cer=0.0435 | wer=0.0667 | pn_recall=0.0000
- lee-eunje/record14.mp4 | pp_on=0 | cer=0.0435 | wer=0.0667 | pn_c=0.0 | pn_recall=0.0000
ref: 볼륨을 십으로 맞추고 화면 모드를 영화 모드로 변경해줘
hyp_raw: 볼륨 십으로 맞추고 화면모드를 영화모드로 변경해줘
ref_pp: []
hyp_pp: []

- lee-eunje/record8.mp4 | cer=0.0000 | wer=0.0000 | pn_recall=1.0000
- lee-eunje/record8.mp4 | pp_on=0 | cer=0.0000 | wer=0.0000 | pn_c=0.0 | pn_recall=1.0000
ref: 유튜브에서 와이티엔 뉴스 채널의 라이브 방송을 실시간으로 재생해줘
hyp_raw: 유튜브에서 와이티엔 뉴스 채널의 라이브 방송을 실시간으로 재생해줘
ref_pp: ['유튜브', '와이티엔']
hyp_pp: ['유튜브', '와이티엔']

- lee-eunje/record0.mp4 | cer=0.0000 | wer=0.0000 | pn_recall=1.0000
- lee-eunje/record0.mp4 | pp_on=0 | cer=0.0000 | wer=0.0000 | pn_c=0.0 | pn_recall=1.0000
ref: 넷플릭스에서 이사랑도통역이되나요 내가 보던 부분에서 재생해줘
hyp_raw: 넷플릭스에서 이 사랑도 통역이 되나요 내가 보던 부분에서 재생해줘
ref_pp: ['넷플릭스', '이사랑도통역이되나요']
hyp_pp: ['넷플릭스', '이사랑도통역이되나요']

- lee-eunje/record1.mp4 | cer=0.0417 | wer=0.1538 | pn_recall=1.0000
- lee-eunje/record1.

KeyboardInterrupt: 